In [ ]:
from open_dataset_store import quick_start
import pandas as pd
import numpy as np

import pandas as pd


In [3]:
store = quick_start('.', backend='local')
sum = store.summary()


Store initialised at: . (Backend: local)
📊 Dataset Store Summary
Base Directory : .
Backend        : local
------------------------------
Entities (Total: 2)
  - zones: 2
------------------------------
Entries (Total: 2)
  - experiments: 2


In [28]:
store.list_entries('experiments')

,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files,processed_metadata
0,entry_0001,zone_001,1783344483,Senosr Data Test,raw_data/experiments/entry_0001_zone_001_17833...,{'refactored_data': 'processed_data/experiment...,{'refactored_data': {}}
1,entry_0002,zone_001,1783344565,"Test CO2 variation, single person come in and ...",raw_data/experiments/entry_0002_zone_001_17833...,{'refactored_data': 'processed_data/experiment...,{'refactored_data': {}}
2,entry_0003,zone_002,1785482758,Test 1,raw_data/experiments/entry_0005_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
3,entry_0004,zone_002,1785483118,Test 2,raw_data/experiments/entry_0006_zone_002_17854...,{'controller_evaluation': 'processed_data/expe...,{'controller_evaluation': {'description': 'Off...
4,entry_0005,zone_002,1785483802,Test 3,raw_data/experiments/entry_0005_zone_002_17854...,{},NaN
5,entry_0006,zone_002,1785483851,Test 3 Take 2,raw_data/experiments/entry_0006_zone_002_17854...,{},NaN


In [32]:
csv_path = "./3_2.csv"

manual_occupancy = {
    "12:07:00": 0,
    "12:13:00": 1,
    "12:26:00": 2,
    "12:31:00": 1,
    "12:41:00": 0,
}

In [34]:
import pandas as pd


df_raw = pd.read_csv(csv_path)

def add_occupancy_and_localize_time(df, occupancy_schedule, tz='Asia/Kolkata'):
    """
    Converts 'timestamp' to local time and injects a forward-filled 
    'actual_occupancy' column based on a provided manual schedule.
    """
    # 1. Ensure timestamp is timezone aware and convert to local timezone
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df['timestamp'] = df['timestamp'].dt.tz_convert(tz)
    
    # 2. Create a schedule dataframe
    schedule_df = pd.DataFrame(list(occupancy_schedule.items()), columns=['time_str', 'actual_occupancy'])
    
    # Extract the base date from the dataset (assuming 1-day experiment)
    exp_date = df['timestamp'].dt.date.iloc[0]
    
    # [FIXED] Use str(exp_date) instead of exp_date.astype(str)
    schedule_df['timestamp'] = pd.to_datetime(str(exp_date) + ' ' + schedule_df['time_str'])
    schedule_df['timestamp'] = schedule_df['timestamp'].dt.tz_localize(tz)
    schedule_df = schedule_df.sort_values('timestamp')
    
    # 3. Sort main data and merge using nearest backward match (holds previous value)
    df = df.sort_values('timestamp')
    df = pd.merge_asof(df, schedule_df, on='timestamp', direction='backward')
    
    # 4. Fill any timestamps that occurred before the first schedule entry with 0
    df['actual_occupancy'] = df['actual_occupancy'].fillna(0).astype(int)
    
    return df

df_processed = add_occupancy_and_localize_time(df_raw, manual_occupancy)
display(df_processed.head())


,timestamp,outside_t,outside_h,outside_c,outside_p,outside_a,outside_v,room_1_t,room_1_h,room_1_c,...,heated_a,heated_v,mixer,fan,flowrate,coolerState,heaterState,humidifierState,time_str,actual_occupancy
0,2026-07-31 12:07:56.999000+05:30,30.6,67.6,458,0,1,51,22.1,65.2,0,...,0,0,100,57,0,0,0,0,12:07:00,0
1,2026-07-31 12:08:02.060000+05:30,30.5,67.3,438,0,1,41,22.1,65.3,0,...,0,0,100,57,0,0,0,0,12:07:00,0
2,2026-07-31 12:08:06.996000+05:30,30.5,67.6,425,0,1,35,22.1,65.3,0,...,0,0,100,57,0,0,0,0,12:07:00,0
3,2026-07-31 12:08:12.002000+05:30,30.5,67.3,409,0,1,28,22.1,65.3,0,...,0,0,100,57,0,0,0,0,12:07:00,0
4,2026-07-31 12:08:16.995000+05:30,30.5,67.1,419,0,1,32,22.1,65.3,0,...,0,0,100,57,0,0,0,0,12:07:00,0


In [35]:
entry_id = store.create_entry_from_df(
    entry_type="experiments",
    df=df_processed,
    entity_id="zone_002",
    description="Test 3",
)



✅ Entry 'entry_0006' created. File saved as entry_0006_zone_002_1785484346_data.csv


In [31]:
store.delete_entry(entry_id='entry_0006', entry_type="experiments",)

  - Deleted raw file: raw_data/experiments/entry_0006_zone_002_1785483851_data.csv
  - Deleted processed [controller_evaluation]: processed_data/experiments/controller_evaluation/entry_0006_controller_evaluation.parquet
✅ Entry 'entry_0006' completely removed.


'entry_0006'